<a href="https://colab.research.google.com/github/ablamitovroman/data-analysis-practice/blob/main/Binance_Wallet_Analyse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🟦 Ячейка 1: Подключение Google Диска, авторизация и создание 'Wallet Übersicht'

In [ ]:
import os
import gspread
import pandas as pd
from google.colab import auth, drive
from google.auth import default
from googleapiclient.discovery import build

# 1. Подключаем Google Диск
drive.mount('/content/drive', force_remount=False)

# 2. Авторизация
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)

# 3. Поиск / Создание целевой папки /Finance/Binance
def get_or_create_folder(folder_path="Finance/Binance"):
    parts = folder_path.strip("/").split("/")
    parent_id = 'root'
    for part in parts:
        query = f"'{parent_id}' in parents and name = '{part}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
        res = drive_service.files().list(q=query, fields="files(id, name)").execute()
        files = res.get('files', [])
        if files:
            parent_id = files[0]['id']
        else:
            folder_metadata = {
                'name': part,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [parent_id]
            }
            folder = drive_service.files().create(body=folder_metadata, fields='id').execute()
            parent_id = folder.get('id')
    return parent_id

target_folder_id = get_or_create_folder("Finance/Binance")
DATEI_NAME = "Wallet Übersicht"

# 4. Открытие / Создание файла строго внутри папки Finance/Binance
query = f"'{target_folder_id}' in parents and name = '{DATEI_NAME}' and mimeType = 'application/vnd.google-apps.spreadsheet' and trashed = false"
res = drive_service.files().list(q=query, fields="files(id, name)").execute()
files = res.get('files', [])

if files:
    sh = gc.open_by_key(files[0]['id'])
    print(f"Файл '{DATEI_NAME}' успешно открыт по адресу: /content/drive/MyDrive/Finance/Binance/{DATEI_NAME}.gsheet")
else:
    sh = gc.create(DATEI_NAME, folder_id=target_folder_id)
    print(f"Создан новый файл '{DATEI_NAME}' в папке /Finance/Binance/")

print("Прямая ссылка на Google Таблицу:", sh.url)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 🟩 Ячейка 2: Сканирование CSV, слияние с Master_Input и Дедупликация


In [ ]:
import os
import glob
import pandas as pd

# ==============================================================================
# 🟩 ЯЧЕЙКА 2: Сканирование CSV и обновление Master_Input
# ==============================================================================

FOLDER_PATH = "/content/drive/MyDrive/Finance/Binance"
MASTER_SHEET_NAME = "Master_Input"

csv_files = glob.glob(os.path.join(FOLDER_PATH, "*.csv"))

# 1. Чтение сохраненной базы из Google Таблицы
try:
    ws_master = sh.worksheet(MASTER_SHEET_NAME)
    records = ws_master.get_all_records()
    df_master_old = pd.DataFrame(records) if records else pd.DataFrame()
    if not df_master_old.empty:
        print(f"📊 В Google Таблице найдена Мастер-база: {len(df_master_old)} строк.")
except Exception:
    ws_master = sh.add_worksheet(MASTER_SHEET_NAME, rows="5000", cols="20")
    df_master_old = pd.DataFrame()

# 2. Сканирование и объединение CSV файлов
df_raw_list = []
if not df_master_old.empty:
    df_raw_list.append(("Master_Old", df_master_old))

if csv_files:
    print(f"📁 Найдено CSV файлов в папке: {len(csv_files)}")
    for f in csv_files:
        try:
            temp_df = pd.read_csv(f)
            df_raw_list.append((os.path.basename(f), temp_df))
            print(f"  • Прочитан: {os.path.basename(f)} ({len(temp_df)} строк)")
        except Exception as e:
            print(f"  ❌ Ошибка чтения {f}: {e}")
else:
    print("ℹ️ CSV-файлов в папке не найдено. Работаем с сохраненной Мастер-базой.")

# 3. Дедупликация и хронологическая сортировка
processed_dfs = []
if df_raw_list:
    for source_name, df_item in df_raw_list:
        if df_item.empty:
            continue
        df_curr = df_item.copy()
        time_col = 'UTC_Time' if 'UTC_Time' in df_curr.columns else ('Time' if 'Time' in df_curr.columns else df_curr.columns[0])

        # Нормализация даты
        dt_series = pd.to_datetime(df_curr[time_col], format='mixed', dayfirst=True, errors='coerce')
        df_curr[time_col] = dt_series.dt.strftime('%Y-%m-%d %H:%M:%S')

        # Формирование дедупликационного ключа
        norm_time = df_curr[time_col].astype(str).str.strip()
        norm_change = pd.to_numeric(df_curr['Change'], errors='coerce').round(8).astype(str)
        norm_op = df_curr['Operation'].astype(str).str.strip()
        norm_coin = df_curr['Coin'].astype(str).str.strip()

        # Порядковый индекс повторений внутри конкретного файла
        df_curr['file_seq'] = df_curr.groupby([norm_time, norm_op, norm_coin, norm_change]).cumcount()
        df_curr['dedup_key'] = (
            norm_time + "_" +
            norm_op + "_" +
            norm_coin + "_" +
            norm_change + "_" +
            df_curr['file_seq'].astype(str)
        )
        processed_dfs.append(df_curr)

    df_combined = pd.concat(processed_dfs, ignore_index=True)
    df_master_clean = df_combined.drop_duplicates(subset=['dedup_key'], keep='first').copy()
    df_master_clean = df_master_clean.drop(columns=['dedup_key', 'file_seq'])

    # Сортировка по времени
    time_col = 'UTC_Time' if 'UTC_Time' in df_master_clean.columns else 'Time'
    df_master_clean['Time_dt'] = pd.to_datetime(df_master_clean[time_col])
    df_master_clean = df_master_clean.sort_values(by='Time_dt', ascending=True).drop(columns=['Time_dt'])

    # Запись в RAW формате в Google Таблицу
    ws_master.clear()
    ws_master.update(
        values=[df_master_clean.columns.tolist()] + df_master_clean.astype(str).values.tolist(),
        range_name="A1",
        value_input_option="RAW"
    )

    df_verlauf = df_master_clean.copy()
    print(f"\n✅ Синхронизация завершена! Уникальных сделок в Мастер-базе: {len(df_verlauf)}")
else:
    print("❌ Данные отсутствуют.")

### 🟨 Ячейка 3: Фильтрация и первичная очистка данных

In [ ]:
import pandas as pd

# ==============================================================================
# 🟨 ЯЧЕЙКА 3: Фильтрация неторговых операций
# ==============================================================================

ignoriere_ops = [
    'Simple Earn Flexible Subscription',
    'Simple Earn Flexible Redemption',
    'Simple Earn Flexible Interest',
    'Simple Earn Locked Subscription',
    'Simple Earn Locked Redemption',
    'Simple Earn Locked Interest',
    'Flexible Savings Subscription',
    'Flexible Savings Redemption',
    'Flexible Savings Interest',
    'ETH 2.0 Staking',
    'Vega - Funds Transfer',
    'Fund Transfer',
    'Cashback Voucher'
]

# 1. Фильтрация служебных записей Binance
df_input_bereinigt = df_verlauf[~df_verlauf['Operation'].isin(ignoriere_ops)].copy()

# 2. Удаление неиспользуемых полей
cols_to_drop = ['User_ID', 'User ID', 'Remark', 'Account']
df_input_bereinigt = df_input_bereinigt.drop(columns=[col for col in cols_to_drop if col in df_input_bereinigt.columns])

print(f"Всего записей в Master_Input: {len(df_verlauf)}")
print(f"Чистых торговых записей для портфеля: {len(df_input_bereinigt)}")

### 🟧 Ячейка 4: Выгрузка сырых данных (Input) и создание вкладки 'Telegram_Prices'

In [ ]:
import gspread

# ==============================================================================
# 🟧 ЯЧЕЙКА 4: Выгрузка Input и настройка Telegram_Prices
# ==============================================================================

# 1. Выгрузка чистых торговых транзакций на лист Input
try:
    ws_input = sh.worksheet("Input")
except gspread.WorksheetNotFound:
    ws_input = sh.add_worksheet("Input", rows="1000", cols="20")

daten_binance = [df_input_bereinigt.columns.tolist()] + df_input_bereinigt.astype(str).values.tolist()
ws_input.clear()
ws_input.update(values=daten_binance, range_name="A1", value_input_option="RAW")
print("1. Лист 'Input' успешно обновлен из Master_Input!")

# 2. Создание или проверка шаблона для актуальных цен из Telegram
sample_tg_text = [
    ["Пара", "Отсл.", "Актуал.", "Стоимость", "Изменение"],
    ["BTC USDC", "97740.2", "84426.2", "-768.26", "🔴-13.62%"],
    ["ETH USDC", "3398.72", "2659.00", "-401.54", "🔴-21.76%"],
    ["SOL USDC", "171.84", "117.73", "-70.61", "🔴-31.49%"],
    ["LINK USDC", "20.01", "13.59", "-59.30", "🔴-32.11%"],
    ["ADA USDC", "0.59", "0.24", "-82.95", "🔴-59.70%"],
    ["SUI USDC", "4.86", "1.12", "-155.41", "🔴-76.94%"],
    ["XRP USDC", "1.75", "1.47", "-13.57", "🔴-15.83%"],
    ["GRAM USDC", "2.12", "1.49", "-21.19", "🔴-29.76%"],
    ["BNB USDC", "671.47", "764.67", "+7.98", "🟢+13.88%"],
    ["XLM USDC", "0.16", "0.21", "+12.23", "🟢+29.03%"],
    ["AIXBT USD", "1.62", "0.02", "-0.32", "🔴-98.65%"]
]

try:
    ws_prices = sh.worksheet("Telegram_Prices")
except gspread.WorksheetNotFound:
    ws_prices = sh.add_worksheet("Telegram_Prices", rows="50", cols="10")
    ws_prices.update(values=sample_tg_text, range_name="A1", value_input_option="USER_ENTERED")
    print("2. Лист 'Telegram_Prices' создан!")

# 3. Безопасное удаление стандартного пустого листа
for default_name in ["Tabellenblatt1", "Sheet1"]:
    try:
        default_ws = sh.worksheet(default_name)
        sh.del_worksheet(default_ws)
        print(f"3. Удален стандартный пустой лист '{default_name}'")
    except Exception:
        pass

### 🟪 Ячейка 5: Расчет портфеля (Подход 2), Резерва USDC и Триггеров

In [ ]:
import pandas as pd
import requests
import gspread

# ==============================================================================
# 🟪 ЯЧЕЙКА 5: Расчет портфеля (Подход 2), Резерва USDC и Триггеров
# ==============================================================================

def load_prices_from_telegram_sheet(sh):
    """
    Извлекает рыночные котировки монет строго из листа Telegram_Prices.
    """
    prices = {
        'USDT': 1.0, 'USDC': 1.0, 'EUR': 1.08,
        'BTC': 84426.2, 'ETH': 2659.0, 'SOL': 117.73, 'LINK': 13.59,
        'XRP': 1.47, 'BNB': 764.67, 'ADA': 0.24, 'XLM': 0.21,
        'SUI': 1.12, 'GRAM': 1.49, 'ACT': 0.345, 'AIXBT': 0.02,
        'LINEA': 1.05, 'OPEN': 0.028, 'SOMI': 0.0045
    }
    sheet_names = ["Telegram_Prices", "Telegram_Input", "Telegram"]
    ws = None
    for s_name in sheet_names:
        try:
            ws = sh.worksheet(s_name)
            break
        except gspread.WorksheetNotFound:
            continue

    if ws is None:
        print("Предупреждение: Лист с ценами Telegram не найден. Использованы резервные котировки.")
        return prices

    try:
        rows = ws.get_all_values()
        for row in rows:
            if not row or not any(row): continue
            raw_line = " ".join([str(c).strip() for c in row if str(c).strip() != ""])
            if "─" in raw_line or "ИТОГО" in raw_line.upper() or "ПАРА" in raw_line.upper():
                continue
            tokens = raw_line.split()
            if len(tokens) < 3: continue
            coin = tokens[0].upper()
            if coin == 'TON': coin = 'GRAM'
            actual_price_str = tokens[3] if len(tokens) >= 4 and tokens[1].upper() in ['USDC', 'USDT', 'USD', 'EUR'] else tokens[2]
            if actual_price_str:
                clean_p_str = actual_price_str.replace(',', '.').replace('$', '').replace('€', '').strip()
                try:
                    p_val = float(clean_p_str)
                    if p_val > 0: prices[coin] = p_val
                except ValueError: pass
        print(f"Цены успешно загружены из листа '{ws.title}'!")
    except Exception as e:
        print(f"Ошибка при парсинге цен из Telegram: {e}")
    return prices

eur_rate_cache = {}
def get_historical_eur_rate(timestamp_val):
    """
    Запрашивает точный исторический курс EUR/USDT на момент совершения сделки.
    """
    ts_str = str(timestamp_val)
    if ts_str in eur_rate_cache: return eur_rate_cache[ts_str]
    try:
        dt = pd.to_datetime(timestamp_val)
        unix_ts = int(dt.timestamp())
        url = f"https://min-api.cryptocompare.com/data/pricehistorical?fsym=EUR&tsyms=USDT&ts={unix_ts}"
        res = requests.get(url, timeout=3).json()
        if 'EUR' in res and 'USDT' in res['EUR']:
            rate = float(res['EUR']['USDT'])
            eur_rate_cache[ts_str] = rate
            return rate
    except Exception: pass
    return 1.08

time_col = 'UTC_Time' if 'UTC_Time' in df_input_bereinigt.columns else 'Time'
fiat_coins = ['EUR', 'USDC', 'USDT']

df_input_bereinigt['Time_dt'] = pd.to_datetime(df_input_bereinigt[time_col])
df_sorted = df_input_bereinigt.sort_values(by='Time_dt', ascending=True).copy()
df_sorted['Coin'] = df_sorted['Coin'].replace({'TON': 'GRAM'})

def calc_usdc_value(row):
    val = row['Change']
    coin = row['Coin']
    if coin == 'EUR':
        return val * get_historical_eur_rate(row[time_col])
    elif coin in ['USDC', 'USDT']:
        return val
    return 0.0

df_sorted['USDC_Change'] = df_sorted.apply(calc_usdc_value, axis=1)
df_sorted['Time_Window'] = df_sorted['Time_dt'].dt.floor('1min')

all_crypto_coins = sorted(list(set(df_sorted[~df_sorted['Coin'].isin(fiat_coins)]['Coin'])))
market_prices = load_prices_from_telegram_sheet(sh)

portfolio_state = {coin: {'units': 0.0, 'cost_basis': 0.0, 'realized_pnl': 0.0} for coin in all_crypto_coins}

grouped_windows = df_sorted.groupby('Time_Window')
for ts_window, group in grouped_windows:
    fiat_spent = abs(group[(group['Coin'].isin(fiat_coins)) & (group['USDC_Change'] < 0)]['USDC_Change'].sum())
    fiat_received = group[(group['Coin'].isin(fiat_coins)) & (group['USDC_Change'] > 0)]['USDC_Change'].sum()
    crypto_gains = group[(~group['Coin'].isin(fiat_coins)) & (group['Change'] > 0)]
    crypto_sold = group[(~group['Coin'].isin(fiat_coins)) & (group['Change'] < 0)]

    if fiat_spent > 0 and not crypto_gains.empty:
        total_est_val = sum(row['Change'] * market_prices.get(row['Coin'], 1.0) for _, row in crypto_gains.iterrows())
        for _, row in crypto_gains.iterrows():
            coin = row['Coin']
            units_bought = row['Change']
            coin_est_val = units_bought * market_prices.get(coin, 1.0)
            share = (coin_est_val / total_est_val) if total_est_val > 0 else (1.0 / len(crypto_gains))
            portfolio_state[coin]['units'] += units_bought
            portfolio_state[coin]['cost_basis'] += fiat_spent * share

    elif fiat_received > 0 and not crypto_sold.empty:
        total_est_sold_val = sum(abs(row['Change']) * market_prices.get(row['Coin'], 1.0) for _, row in crypto_sold.iterrows())
        for _, row in crypto_sold.iterrows():
            coin = row['Coin']
            units_sold = abs(row['Change'])
            coin_est_val = units_sold * market_prices.get(coin, 1.0)
            share = (coin_est_val / total_est_sold_val) if total_est_sold_val > 0 else (1.0 / len(crypto_sold))
            allocated_revenue = fiat_received * share
            c_units = portfolio_state[coin]['units']
            c_cost = portfolio_state[coin]['cost_basis']
            avg_p = (c_cost / c_units) if c_units > 0 else 0.0
            cost_of_sold = units_sold * avg_p
            pnl_realized = allocated_revenue - cost_of_sold
            portfolio_state[coin]['realized_pnl'] += pnl_realized
            portfolio_state[coin]['cost_basis'] = max(0.0, c_cost - cost_of_sold)
            portfolio_state[coin]['units'] = max(0.0, c_units - units_sold)

    elif fiat_spent == 0 and not crypto_gains.empty and not crypto_sold.empty:
        transferred_cost = 0.0
        for _, row in crypto_sold.iterrows():
            coin_out = row['Coin']
            units_sold = abs(row['Change'])
            c_units = portfolio_state[coin_out]['units']
            c_cost = portfolio_state[coin_out]['cost_basis']
            avg_p = (c_cost / c_units) if c_units > 0 else market_prices.get(coin_out, 0.0)
            sold_cost = units_sold * avg_p
            transferred_cost += sold_cost
            portfolio_state[coin_out]['cost_basis'] = max(0.0, c_cost - sold_cost)
            portfolio_state[coin_out]['units'] = max(0.0, c_units - units_sold)
        for _, row in crypto_gains.iterrows():
            coin_in = row['Coin']
            portfolio_state[coin_in]['units'] += row['Change']
            portfolio_state[coin_in]['cost_basis'] += transferred_cost / len(crypto_gains)

portfolio_rows = []
for coin, st in portfolio_state.items():
    units = st['units']
    cost = st['cost_basis']
    realized_pnl = st['realized_pnl']
    if units > 0.000001 or abs(realized_pnl) > 0.01:
        avg_price = (cost / units) if units > 0 else 0.0
        curr_price = market_prices.get(coin, 0.0)
        curr_value = units * curr_price
        unrealized_pnl = curr_value - cost if units > 0 else 0.0
        portfolio_rows.append({
            'Coin': coin, 'Anzahl_Menge': units, 'Investiert_Gesamt': cost,
            'Durchschnittspreis': avg_price, 'Aktueller_Preis': curr_price,
            'Aktueller_Gesamtwert': curr_value, 'Unrealized_PnL': unrealized_pnl,
            'Realized_PnL': realized_pnl, 'Total_PnL': unrealized_pnl + realized_pnl
        })

usdc_reserve_spot = df_sorted[df_sorted['Coin'].isin(fiat_coins)]['USDC_Change'].sum()
interest_ops = [op for op in df_verlauf['Operation'].unique() if 'Interest' in str(op)]
df_earn_interest = df_verlauf[(df_verlauf['Operation'].isin(interest_ops)) & (df_verlauf['Coin'].isin(fiat_coins))].copy()
earn_interest_total = df_earn_interest.apply(calc_usdc_value, axis=1).sum() if not df_earn_interest.empty else 0.0
usdc_reserve = usdc_reserve_spot + earn_interest_total

portfolio_rows.append({
    'Coin': 'USDC (Reserve)', 'Anzahl_Menge': usdc_reserve, 'Investiert_Gesamt': usdc_reserve,
    'Durchschnittspreis': 1.0, 'Aktueller_Preis': 1.0, 'Aktueller_Gesamtwert': usdc_reserve,
    'Unrealized_PnL': 0.0, 'Realized_PnL': earn_interest_total, 'Total_PnL': earn_interest_total
})

df_portfolio_raw = pd.DataFrame(portfolio_rows)
df_crypto_part = df_portfolio_raw[df_portfolio_raw['Coin'] != 'USDC (Reserve)'].sort_values(by='Aktueller_Gesamtwert', ascending=False)
df_reserve_part = df_portfolio_raw[df_portfolio_raw['Coin'] == 'USDC (Reserve)']
df_portfolio = pd.concat([df_crypto_part, df_reserve_part], ignore_index=True)

# Сводный итоговый блок с отступом
blank_row = {col: '' for col in df_portfolio.columns}
summary_row = {
    'Coin': 'ИТОГО ПОРТФЕЛЬ',
    'Anzahl_Menge': '',
    'Investiert_Gesamt': df_portfolio['Investiert_Gesamt'].sum(),
    'Durchschnittspreis': '',
    'Aktueller_Preis': '',
    'Aktueller_Gesamtwert': df_portfolio['Aktueller_Gesamtwert'].sum(),
    'Unrealized_PnL': df_portfolio['Unrealized_PnL'].sum(),
    'Realized_PnL': df_portfolio['Realized_PnL'].sum(),
    'Total_PnL': df_portfolio['Total_PnL'].sum()
}

df_portfolio_export = pd.concat([df_portfolio, pd.DataFrame([blank_row]), pd.DataFrame([summary_row])], ignore_index=True)

def get_or_create_worksheet(sh, title, rows="100", cols="12"):
    for ws in sh.worksheets():
        if ws.title.lower() == title.lower():
            if ws.title != title: ws.update_title(title)
            return ws
    return sh.add_worksheet(title, rows=str(rows), cols=str(cols))

ws_port = get_or_create_worksheet(sh, "Portfolio", rows="100", cols="12")
ws_port.clear()
ws_port.update(values=[df_portfolio_export.columns.tolist()] + df_portfolio_export.values.tolist(), range_name="A1", value_input_option="USER_ENTERED")

df_trigger = pd.DataFrame()
df_trigger['Coin'] = df_portfolio['Coin'].apply(lambda c: c if 'Reserve' in c else f"{c} USDC")
df_trigger['Avg. Price'] = df_portfolio['Durchschnittspreis']
df_trigger['CumUnits'] = df_portfolio['Anzahl_Menge']
df_trigger['Investiert USDC'] = df_portfolio['Investiert_Gesamt']
df_trigger['Aktueller Preis'] = df_portfolio['Aktueller_Preis']
df_trigger['Unrealized Profit'] = df_portfolio['Unrealized_PnL']
df_trigger['Realized Profit'] = df_portfolio['Realized_PnL']
df_trigger['-15%'] = df_trigger.apply(lambda r: r['Avg. Price'] * 0.85 if 'Reserve' not in str(r['Coin']) else 1.0, axis=1)
df_trigger['-25%'] = df_trigger.apply(lambda r: r['Avg. Price'] * 0.75 if 'Reserve' not in str(r['Coin']) else 1.0, axis=1)
df_trigger['50%'] = df_trigger.apply(lambda r: r['Avg. Price'] * 1.50 if 'Reserve' not in str(r['Coin']) else 1.0, axis=1)

ws_trigger = get_or_create_worksheet(sh, "Trigger", rows="100", cols="12")
ws_trigger.clear()
ws_trigger.update(values=[df_trigger.columns.tolist()] + df_trigger.values.tolist(), range_name="A1", value_input_option="USER_ENTERED")

print("Расчет завершен! Листы 'Portfolio' и 'Trigger' успешно обновлены.")

### 🟫 Ячейка 6: Детализация истории по монетам

In [ ]:
import pandas as pd
import gspread

# ==============================================================================
# 🟫 ЯЧЕЙКА 6: Единый лист истории 'Coin' для всех монет
# ==============================================================================

SHEET_NAME = "Coin"

time_col = 'UTC_Time' if 'UTC_Time' in df_input_bereinigt.columns else 'Time'
fiat_coins = ['EUR', 'USDC', 'USDT']

df_input_bereinigt['Time_dt'] = pd.to_datetime(df_input_bereinigt[time_col])
df_sorted = df_input_bereinigt.sort_values(by='Time_dt', ascending=True).copy()
df_sorted['Coin'] = df_sorted['Coin'].replace({'TON': 'GRAM'})

df_sorted['USDC_Change'] = df_sorted.apply(calc_usdc_value, axis=1)
df_sorted['Time_Window'] = df_sorted['Time_dt'].dt.floor('1min')

all_crypto_coins = sorted(list(set(df_sorted[~df_sorted['Coin'].isin(fiat_coins)]['Coin'])))
market_prices = load_prices_from_telegram_sheet(sh)

portfolio_state = {coin: {'units': 0.0, 'cost_basis': 0.0, 'realized_pnl': 0.0} for coin in all_crypto_coins}
coin_histories = {coin: [] for coin in all_crypto_coins}

grouped_windows = df_sorted.groupby('Time_Window')

for ts_window, group in grouped_windows:
    fiat_spent = abs(group[(group['Coin'].isin(fiat_coins)) & (group['USDC_Change'] < 0)]['USDC_Change'].sum())
    fiat_received = group[(group['Coin'].isin(fiat_coins)) & (group['USDC_Change'] > 0)]['USDC_Change'].sum()
    crypto_gains = group[(~group['Coin'].isin(fiat_coins)) & (group['Change'] > 0)]
    crypto_sold = group[(~group['Coin'].isin(fiat_coins)) & (group['Change'] < 0)]

    active_coins_in_group = set(group[~group['Coin'].isin(fiat_coins)]['Coin'])
    before_state = {
        c: {
            'units': portfolio_state[c]['units'],
            'cost': portfolio_state[c]['cost_basis'],
            'pnl': portfolio_state[c]['realized_pnl']
        } for c in active_coins_in_group
    }

    if fiat_spent > 0 and not crypto_gains.empty:
        total_est_val = sum(row['Change'] * market_prices.get(row['Coin'], 1.0) for _, row in crypto_gains.iterrows())
        for _, row in crypto_gains.iterrows():
            coin = row['Coin']
            units_bought = row['Change']
            coin_est_val = units_bought * market_prices.get(coin, 1.0)
            share = (coin_est_val / total_est_val) if total_est_val > 0 else (1.0 / len(crypto_gains))
            portfolio_state[coin]['units'] += units_bought
            portfolio_state[coin]['cost_basis'] += fiat_spent * share

    elif fiat_received > 0 and not crypto_sold.empty:
        total_est_sold_val = sum(abs(row['Change']) * market_prices.get(row['Coin'], 1.0) for _, row in crypto_sold.iterrows())
        for _, row in crypto_sold.iterrows():
            coin = row['Coin']
            units_sold = abs(row['Change'])
            coin_est_val = units_sold * market_prices.get(coin, 1.0)
            share = (coin_est_val / total_est_sold_val) if total_est_sold_val > 0 else (1.0 / len(crypto_sold))
            allocated_revenue = fiat_received * share
            c_units = portfolio_state[coin]['units']
            c_cost = portfolio_state[coin]['cost_basis']
            avg_p = (c_cost / c_units) if c_units > 0 else 0.0
            cost_of_sold = units_sold * avg_p
            pnl_realized = allocated_revenue - cost_of_sold
            portfolio_state[coin]['realized_pnl'] += pnl_realized
            portfolio_state[coin]['cost_basis'] = max(0.0, c_cost - cost_of_sold)
            portfolio_state[coin]['units'] = max(0.0, c_units - units_sold)

    elif fiat_spent == 0 and not crypto_gains.empty and not crypto_sold.empty:
        transferred_cost = 0.0
        for _, row in crypto_sold.iterrows():
            coin_out = row['Coin']
            units_sold = abs(row['Change'])
            c_units = portfolio_state[coin_out]['units']
            c_cost = portfolio_state[coin_out]['cost_basis']
            avg_p = (c_cost / c_units) if c_units > 0 else market_prices.get(coin_out, 0.0)
            sold_cost = units_sold * avg_p
            transferred_cost += sold_cost
            portfolio_state[coin_out]['cost_basis'] = max(0.0, c_cost - sold_cost)
            portfolio_state[coin_out]['units'] = max(0.0, c_units - units_sold)

        for _, row in crypto_gains.iterrows():
            coin_in = row['Coin']
            portfolio_state[coin_in]['units'] += row['Change']
            portfolio_state[coin_in]['cost_basis'] += transferred_cost / len(crypto_gains)

    for coin in active_coins_in_group:
        coin_rows = group[group['Coin'] == coin]
        if coin_rows.empty: continue

        change_units = coin_rows['Change'].sum()
        op_type = ", ".join(coin_rows['Operation'].unique())
        time_str = coin_rows['Time_dt'].iloc[0].strftime('%Y-%m-%d %H:%M:%S')

        units_after = portfolio_state[coin]['units']
        cost_after = portfolio_state[coin]['cost_basis']
        pnl_after = portfolio_state[coin]['realized_pnl']

        cost_before = before_state[coin]['cost']
        pnl_before = before_state[coin]['pnl']

        trade_cost_or_rev = cost_after - cost_before
        trade_pnl = pnl_after - pnl_before
        avg_price_now = (cost_after / units_after) if units_after > 0 else 0.0

        coin_histories[coin].append([
            time_str, op_type, change_units, units_after,
            trade_cost_or_rev, cost_after, avg_price_now,
            trade_pnl, pnl_after
        ])

all_sheet_rows = []
headers = [
    'Datum_Uhrzeit', 'Operation', 'Change_Menge', 'CumUnits_Rest',
    'Investiert_Trade_USDC', 'Cost_Basis_Gesamt', 'Avg_Price_Entry',
    'Realized_PnL_Trade', 'Cum_Realized_PnL'
]

# Сортировка монет strictly matching Valuation Order in Trigger / Portfolio
ordered_coins = sorted(
    all_crypto_coins,
    key=lambda c: portfolio_state[c]['units'] * market_prices.get(c, 0.0),
    reverse=True
)

for coin in ordered_coins:
    history_records = coin_histories[coin]
    if not history_records: continue

    # Заголовок с дефисами (исключает ошибку формул)
    title_row = [f"-------------------- {coin} --------------------"] + [""] * 8
    all_sheet_rows.append(title_row)
    all_sheet_rows.append(headers)
    all_sheet_rows.extend(history_records)
    all_sheet_rows.append([""] * 9)

ws_coin = get_or_create_worksheet(sh, SHEET_NAME, rows=str(max(2000, len(all_sheet_rows) + 50)), cols="12")
ws_coin.clear()
ws_coin.update(values=all_sheet_rows, range_name="A1", value_input_option="USER_ENTERED")

print(f"Все монеты выгружены на единый лист '{SHEET_NAME}'. Себестоимость на 100% совпадает с 'Portfolio'!")